# 01 — Data Cleaning

## Objective
Handle missing values in the raw Kaggle housing dataset before modeling.

## Missing Value Strategy
| Column(s) | Strategy | Reason |
|---|---|---|
| PoolQC, MiscFeature, Alley, Fence | Drop | 80–99% missing, near-zero signal |
| FireplaceQu, Garage*, Bsmt*, MasVnrType | Fill → "None" | Missing means feature doesn't exist |
| LotFrontage | Median imputation | Skewed distribution, value was unrecorded |
| GarageYrBlt, MasVnrArea | Fill → 0 | Missing means no garage/veneer, area is zero |
| Electrical | Mode imputation | Only 1 missing value |

In [32]:
import pandas as pd

df = pd.read_csv("../data/raw/housing.csv")

print(df.shape)
print(df.dtypes.value_counts())
print("\nMissing values (columns with nulls only):")
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
#split missing into numeric vs categorical
numeric_missing = df.select_dtypes(include=['int64', 'float64']).isnull().sum()
numeric_missing = numeric_missing[numeric_missing > 0].sort_values(ascending=False)
categorical_missing = df.select_dtypes(include=['object']).isnull().sum()
categorical_missing = categorical_missing[categorical_missing > 0].sort_values(ascending=False)

print(missing)
print("Numeric columns with missing values:")
print(numeric_missing)

print("\nCategorical columns with missing values:")
print(categorical_missing)

(1460, 81)
object     43
int64      35
float64     3
Name: count, dtype: int64

Missing values (columns with nulls only):
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64
Numeric columns with missing values:
LotFrontage    259
GarageYrBlt     81
MasVnrArea       8
dtype: int64

Categorical columns with missing values:
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
GarageFinish      81
GarageQual        81
GarageType        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtFinType1      37
BsmtCond          37
El

### Data Observations
Group 1:

    PoolQC       1453 / 1460 missing = 99.5%
    MiscFeature  1406 / 1460 missing = 96.3%
    Alley        1369 / 1460 missing = 93.8%
    Fence        1179 / 1460 missing = 80.8%

These columns are so sparse that they can't teach the model anything meaningful. For `PoolQC`, `Alley`, and `Fence` it could mean that they dont have it but the data is so small that keeping them wouldn't make a substantial difference in training the model

Decision: Drop these

Group 2:

    MasVnrType    872 missing
    FireplaceQu   690 missing
    GarageType     81 missing
    GarageFinish   81 missing
    GarageQual     81 missing
    GarageCond     81 missing
    BsmtExposure   38 missing
    BsmtFinType2   38 missing
    BsmtQual       37 missing
    BsmtCond       37 missing
    BsmtFinType1   37 missing

For these columns, a missing value almost certainly means that the house does not have the feature so `"None"` would be a good replacement for `Null` and then it can be encoded

Group 3:

    LotFrontage  259 missing
    GarageYrBlt   81 missing
    MasVnrArea     8 missing

`LotFrontage` - linearfeet of street connected to the property. Missing likely means that it wasn't recorded and not zero. **Median imputation** would be the correct choice since this datas distribution is right skewed


`GarageYrBlt` - Of the house has no garage (Which was established that 81 houses had no garage (group2) ) this should just be 0 and not median

`MasVnrArea` - Only 8 missing, pirs with `MasVnrType`. Fill with 0 since no masonary venner means area is zero.

Group 4:

    Electrical  1 missing

One row is missing just filll it with the mode (most frequent value).

In [33]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/housing.csv")

# ── 1. Drop high-sparsity columns ──────────────────────────────
# These columns are 80–99% missing and carry no meaningful signal
drop_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence']
df = df.drop(columns=drop_cols)

# ── 2. Fill categorical columns where missing means "None" ──────
# Missing here means the house doesn't have that feature
none_fill_cols = [
    'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'MasVnrType'
]
for col in none_fill_cols:
    df[col] = df[col].fillna('None')

# ── 3. Fill numeric columns ─────────────────────────────────────
# LotFrontage: skewed distribution, median is robust to outliers
df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())

# GarageYrBlt: missing = no garage, so 0 is correct
df['GarageYrBlt'] = df['GarageYrBlt'].fillna(0)

# MasVnrArea: missing = no masonry veneer, area is 0
df['MasVnrArea'] = df['MasVnrArea'].fillna(0)

# ── 4. Fill Electrical with mode ────────────────────────────────
# Only 1 missing value, most frequent category is the safe choice
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])

# ── Verify no missing values remain ────────────────────────────
remaining = df.isnull().sum()
remaining = remaining[remaining > 0]
print("Remaining nulls:", remaining if len(remaining) > 0 else "None ✓")

Remaining nulls: None ✓


In [34]:
# Export processed data to processed folder
df.to_csv('../data/processed/housing_processed.csv', index=False)